--------------------------------------------------------Part 1: Agent Reasoning Patterns----------------------------------------------------------

1.1 ReAct (Reason + Act) Pattern
Simple Explanation: The agent thinks out loud → takes action → sees result → thinks again → repeats until done.

In [2]:
# Cell 1: Install required packages


# Cell 2: Setup your API key
import os

from langchain_groq import ChatGroq

# Set your OpenAI API key
llm = ChatGroq(
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

# Cell 3: Simple ReAct implementation
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    question: str
    thoughts: List[str]
    actions: List[str]
    observations: List[str]
    answer: str

def reason(state):
    """Think about what to do next"""
    prompt = f"""
    Question: {state['question']}
    Previous thoughts: {state['thoughts']}
    
    What's your next thought? (Be specific about what action to take)
    """
    response = llm.invoke(prompt)
    state['thoughts'].append(response.content)
    return state

def act(state):
    """Perform the action (in this case, search knowledge base)"""
    # Simulating an action - in real life this could be API calls, database queries, etc.
    action_prompt = f"""
    Based on this thought: {state['thoughts'][-1]}
    What action would you take? (search, calculate, or final_answer)
    """
    response = llm.invoke(action_prompt)
    state['actions'].append(response.content)
    
    # Simulate observation
    if "search" in response.content.lower():
        observation = "Found: The answer is related to AI agents and their patterns"
        state['observations'].append(observation)
    else:
        state['observations'].append("No action needed yet")
    
    return state

def should_continue(state):
    """Decide if we should continue or stop"""
    # If we have enough info or if the answer is clear
    if len(state['thoughts']) > 3:
        return "finalize"
    return "reason"

def finalize(state):
    """Generate final answer"""
    prompt = f"""
    Based on all these thoughts and observations:
    Thoughts: {state['thoughts']}
    Observations: {state['observations']}
    
    Give a final answer to: {state['question']}
    """
    response = llm.invoke(prompt)
    state['answer'] = response.content
    return state

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("reason", reason)
workflow.add_node("act", act)
workflow.add_node("finalize", finalize)

workflow.set_entry_point("reason")
workflow.add_edge("reason", "act")
workflow.add_conditional_edges(
    "act",
    should_continue,
    {
        "reason": "reason",
        "finalize": "finalize"
    }
)
workflow.add_edge("finalize", END)

app = workflow.compile()

# Cell 4: Run ReAct example
initial_state = {
    "question": "What is 4+4?",
    "thoughts": [],
    "actions": [],
    "observations": [],
    "answer": ""
}

result = app.invoke(initial_state)
print("=== REACT PROCESS ===")
print(f"Thoughts: {result['thoughts']}")
print("===------------------------------------- ==="*2)
print(f"Actions: {result['actions']}")
print("===------------------------------------- ==="*2)
print(f"Observations: {result['observations']}")
print("===------------------------------------- ==="*2)
print(f"\nFinal Answer: {result['answer']}")

=== REACT PROCESS ===
Thoughts: ['My next thought is to perform a basic arithmetic operation to calculate the sum of 4 and 4. I will add the two numbers together to get the result. \n\nAction to take: Calculate 4 + 4 = 8.', 'My next thought is to confirm the calculation by reevaluating the arithmetic operation. I will recheck the addition of 4 and 4 to ensure the result is accurate.\n\nAction to take: Re-calculate 4 + 4 to confirm that the result is indeed 8.', 'My next thought is to verify the calculation by using a different method or approach to confirm the result. I will use mental math to quickly add 4 and 4, and then compare the result to my previous calculations to ensure consistency and accuracy.\n\nAction to take: Use mental math to calculate 4 + 4, and then compare the result to the previous calculations to confirm that the answer is indeed 8.', 'My next thought is to provide the final answer based on the previous calculations and verifications. Since I have already performed

1.2 Chain of Thought (CoT)
Simple Explanation: Force the model to show its work step-by-step before giving an answer.

In [ ]:
# Cell 5: Chain of Thought Implementation
def chain_of_thought(question):
    """Force the model to think step by step"""
    prompt = f"""
    Answer this question step by step. Show all your reasoning:
    
    Question: {question}
    
    Step 1: Let me first understand what's being asked...
    Step 2: Now let me think about the key concepts...
    Step 3: Let me consider the solution...
    Step 4: Now I can formulate my answer...
    
    Final Answer: """
    
    response = llm.invoke(prompt)
    return response.content

# Test CoT
question = "If I have 3 apples and give away 2, then buy 5 more, how many do I have?"
print("=== CHAIN OF THOUGHT ===")
print(chain_of_thought(question))

1.3 Plan-and-Solve with Reflection
Simple Explanation: Make a plan first, then execute it, then have a "critic" check the work.

In [ ]:
# Cell 6: Plan and Solve with Critic
class PlanningState(TypedDict):
    question: str
    plan: str
    solution: str
    feedback: str
    final_answer: str

def create_plan(state):
    """Create a detailed plan"""
    prompt = f"""
    Create a detailed plan to answer this question: {state['question']}
    List specific steps (at least 2-3 steps)
    """
    response = llm.invoke(prompt)
    state['plan'] = response.content
    return state

def solve(state):
    """Execute the plan"""
    prompt = f"""
    Follow this plan to solve the question:
    Plan: {state['plan']}
    Question: {state['question']}
    
    Provide the solution following the plan step by step.
    """
    response = llm.invoke(prompt)
    state['solution'] = response.content
    return state

def criticize(state):
    """Critic reviews the solution"""
    prompt = f"""
    You are a critic. Review this solution:
    Solution: {state['solution']}
    Question: {state['question']}
    
    Point out any errors, missing steps, or areas for improvement.
    """
    response = llm.invoke(prompt)
    state['feedback'] = response.content
    return state

def refine(state):
    """Refine based on criticism"""
    prompt = f"""
    Original solution: {state['solution']}
    Critic feedback: {state['feedback']}
    Question: {state['question']}
    
    Provide an improved final answer incorporating the feedback.
    """
    response = llm.invoke(prompt)
    state['final_answer'] = response.content
    return state

# Build plan-solve-reflect workflow
plan_workflow = StateGraph(PlanningState)
plan_workflow.add_node("plan", create_plan)
plan_workflow.add_node("solve", solve)
plan_workflow.add_node("criticize", criticize)
plan_workflow.add_node("refine", refine)

plan_workflow.set_entry_point("plan")
plan_workflow.add_edge("plan", "solve")
plan_workflow.add_edge("solve", "criticize")
plan_workflow.add_edge("criticize", "refine")
plan_workflow.add_edge("refine", END)

plan_app = plan_workflow.compile()

# Cell 7: Run Plan and Solve
initial_plan_state = {
    "question": "Explain how to optimize a slow SQL query",
    "plan": "",
    "solution": "",
    "feedback": "",
    "final_answer": ""
}

result = plan_app.invoke(initial_plan_state)
print("=== PLAN AND SOLVE WITH REFLECTION ===")
print(f"Plan: {result['plan']}")
print(f"\nSolution: {result['solution']}")
print(f"\nCritic Feedback: {result['feedback']}")
print(f"\nFinal Answer: {result['final_answer']}")

----------------------------------------------------Part 2: Memory Types-----------------------------------------------------------

2.1 Short-Term Memory (Conversation State)
Simple Explanation: Just keeping the conversation history in memory.

In [ ]:
# Cell 8: Short-term Memory Example
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Create short-term memory
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Simulate conversation
print("=== SHORT-TERM MEMORY ===")
print("User: My name is Alice")
print(conversation.predict(input="My name is Alice"))
print("\nUser: What's my name?")
print(conversation.predict(input="What's my name?"))
print("\nUser: What did we just talk about?")
print(conversation.predict(input="What did we just talk about?"))

2.2 Long-Term Memory (Vector Stores)
Simple Explanation: Save information permanently and search for relevant info when needed.

In [ ]:
# Cell 9: Long-term Memory Setup
!pip install chromadb

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter

# Create embeddings and vector store
embeddings = OpenAIEmbeddings()

# Example documents to store
documents = [
    "User Alice prefers Python programming",
    "User Bob likes JavaScript and web development",
    "Alice worked on a machine learning project last month",
    "Bob is learning cloud computing with AWS"
]

# Create vector store (long-term memory)
vectorstore = Chroma.from_texts(
    documents,
    embedding=embeddings,
    collection_name="user_preferences"
)

print("=== LONG-TERM MEMORY ===")
# Search for relevant info
query = "What does Alice like?"
results = vectorstore.similarity_search(query, k=2)
print(f"Query: {query}")
print(f"Found: {results[0].page_content}")
print(f"Found: {results[1].page_content}")

2.3 Episodic Memory (Saving Successful Traces)
Simple Explanation: Remember how you solved problems in the past.

In [ ]:
import os
from typing import Annotated, Dict, List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.store.memory import InMemoryStore

# 1. Define the Episodic Schema (The Few-Shot Event)
class EpisodicMemory(BaseModel):
    user_input: str = Field(description="The user email or prompt received")
    action_taken: str = Field(description="Action taken by the agent (e.g., IGNORE, RESPOND)")
    reasoning: str = Field(description="Why this specific action was taken")

# 2. Define the Graph State
class AgentState(TypedDict):
    email_content: str
    triage_action: str = ""
    explanation: str = ""

# 3. Instantiate the Long-Term Memory Store
# In production, swap InMemoryStore for an external vector/document store like Redis or MongoDB
memory_store = InMemoryStore()

# Seed the memory store with mock episodic data (past user examples)
user_id = "user_123"
namespace = (user_id, "episodic_triage_examples")

memory_store.put(
    namespace, 
    "example_1", 
    EpisodicMemory(
        user_input="Exclusive Offer! Buy Bitcoin now and double your returns in 2 hours!!! Click link.",
        action_taken="IGNORE",
        reasoning="This is clearly a spam promotional email and phishing attempt."
    ).model_dump()
)

memory_store.put(
    namespace, 
    "example_2", 
    EpisodicMemory(
        user_input="Hi, can we schedule our weekly sync for Tuesday at 2 PM instead of Monday?",
        action_taken="RESPOND",
        reasoning="This is a direct, legitimate scheduling inquiry from a coworker."
    ).model_dump()
)

# 4. Define the Agent Node Logic
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def triage_email_node(state: AgentState, config: Dict):
    # Retrieve configuration variables
    configurable = config.get("configurable", {})
    current_user = configurable.get("user_id", "default_user")
    
    # Query Episodic Memory using Vector/Namespace Search
    # Here we look up previous examples tied to the user's workspace
    user_namespace = (current_user, "episodic_triage_examples")
    memories = memory_store.search(user_namespace, limit=3)
    
    # Format the past episodes into a clear list of few-shot examples
    few_shot_context = ""
    for idx, mem in enumerate(memories):
        data = mem.value
        few_shot_context += f"\n--- Past Experience {idx+1} ---\n"
        few_shot_context += f"Past Input: {data['user_input']}\n"
        few_shot_context += f"Action Taken: {data['action_taken']}\n"
        few_shot_context += f"Reasoning: {data['reasoning']}\n"

    # Define dynamic prompt utilizing our episodic memory
    system_prompt = (
        "You are an email triage assistant. Categorize incoming emails as 'RESPOND' or 'IGNORE'.\n"
        "Rely on your past experiences below to match context and logic:\n"
        f"{few_shot_context}\n"
        "Now, analyze the new email and output your response in JSON format matching keys: "
        "'action' (RESPOND/IGNORE) and 'reasoning'."
    )
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"New Email: {state['email_content']}")
    ]
    
    response = llm.with_structured_output(AgentState).invoke(messages)
    return response

# 5. Build and Compile the Graph
workflow = StateGraph(AgentState)
workflow.add_node("triage", triage_email_node)
workflow.add_edge(START, "triage")
workflow.add_edge("triage", END)

# Compile the graph while explicitly passing the memory store
app = workflow.compile(store=memory_store)

# 6. Execute the Graph with Episodic Context
config_params = {"configurable": {"user_id": "user_123"}}

test_email = "⚠️ ALERT: UNUSUAL ACTIVITY detected on your crypto wallet. Click here to verify or lose funds."
inputs = {"email_content": test_email}

result = app.invoke(inputs, config=config_params)

print(f"Email Content: {result['email_content']}\n")
print(f"Decision: {result['triage_action']}")
print(f"Reasoning: {result['explanation']}")


------------------------------------------------------Part 3: Retrieval-Augmented Generation (RAG)-----------------------------------------------

3.1 Chunking and Embeddings
Simple Explanation: Break text into pieces, then turn each piece into numbers.

In [ ]:
# Cell 11: Chunking and Embeddings Example
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample text (pretend it's a 500-page PDF)
long_text = """
The history of AI agents begins with simple rule-based systems.
These systems could only respond to specific inputs with predefined outputs.
As technology evolved, we developed more sophisticated agents.
Modern AI agents can reason, plan, and learn from their environment.
They use large language models to understand and generate human-like text.
The key breakthrough was the ability to use tools and external knowledge.
This allowed agents to go beyond their training data.
Today, agents can browse the web, write code, and make decisions.
The future of AI agents involves even more autonomy and capability.
Researchers are working on making agents more reliable and safe.
One challenge is ensuring agents don't hallucinate or make up facts.
Another is making sure they respect user privacy and security.
"""

print("=== CHUNKING AND EMBEDDINGS ===")
# 1. Create chunks with overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Small for demonstration
    chunk_overlap=30,
    length_function=len,
)

chunks = text_splitter.split_text(long_text)
print(f"Created {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk[:100]}...")
    print(f"  (Length: {len(chunk)} chars)")

# 2. Create embeddings
print("\nCreating embeddings for each chunk...")
embeddings_model = OpenAIEmbeddings()
chunk_embeddings = embeddings_model.embed_documents(chunks)

print(f"Created {len(chunk_embeddings)} embeddings")
print(f"Each embedding has {len(chunk_embeddings[0])} dimensions")
print(f"First 5 values of first embedding: {chunk_embeddings[0][:5]}")

# 3.2 Complete RAG Pipeline
# Simple Explanation: Search through documents, find relevant parts, and use them to answer questions.

from langchain.chains import RetrievalQA

# Create a RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2})
)

# Alternative: Manual RAG pipeline
def manual_rag(query):
    """Manual implementation of RAG"""
    # 1. Search for relevant chunks
    print("1. Searching for relevant documents...")
    relevant_docs = vectorstore.similarity_search(query, k=2)
    
    # 2. Prepare context
    print("2. Preparing context...")
    context = "\n".join([doc.page_content for doc in relevant_docs])
    
    # 3. Generate answer with context
    print("3. Generating answer...")
    prompt = f"""
    Context: {context}
    Question: {query}
    
    Answer the question using only the context provided.
    """
    response = llm.invoke(prompt)
    return response.content

print("=== COMPLETE RAG PIPELINE ===")
query = "What programming languages do users like?"
print(f"Query: {query}")
print("\nManual RAG:")
answer = manual_rag(query)
print(f"Answer: {answer}")

-------------------------------------------------------------Part 4: Advanced Tools and Infrastructure-----------------------------------

4.1 Human-in-the-Loop (HITL)
Simple Explanation: Stop and ask for human approval before taking important actions.

In [ ]:
# Cell 13: Human-in-the-Loop Example
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ApprovalState(TypedDict):
    action: str
    is_approved: bool
    execution_result: str

def propose_action(state):
    """Agent proposes an action"""
    state['action'] = "Send email to all users: 'System update at 2PM'"
    state['is_approved'] = False
    print(f"Proposed Action: {state['action']}")
    return state

def get_human_approval(state):
    """Simulate human approval - in real use, this would be a UI"""
    # In a real system, this would pause and wait for user input
    # Here we simulate with a prompt
    user_input = input("Approve action? (y/n): ")
    state['is_approved'] = user_input.lower() == 'y'
    return state

def execute_action(state):
    """Execute if approved"""
    if state['is_approved']:
        # Simulate execution
        state['execution_result'] = "Email sent successfully!"
    else:
        state['execution_result'] = "Action cancelled by user"
    return state

# Build HITL workflow
hitl_workflow = StateGraph(ApprovalState)
hitl_workflow.add_node("propose", propose_action)
hitl_workflow.add_node("approve", get_human_approval)
hitl_workflow.add_node("execute", execute_action)

hitl_workflow.set_entry_point("propose")
hitl_workflow.add_edge("propose", "approve")
hitl_workflow.add_edge("approve", "execute")
hitl_workflow.add_edge("execute", END)

hitl_app = hitl_workflow.compile()

print("=== HUMAN-IN-THE-LOOP ===")
hitl_app.invoke({})

4.2 State Management with Checkpointing
Simple Explanation: Save the state so you can pause, resume, or debug later.

In [ ]:
# Cell 14: State Management with Checkpointing
from langgraph.checkpoint import MemorySaver
import json

class StateManager:
    def __init__(self):
        self.checkpoints = {}
        self.current_checkpoint = None
    
    def save_checkpoint(self, state, label):
        """Save the current state"""
        checkpoint = {
            "state": state,
            "timestamp": datetime.now().isoformat(),
            "label": label
        }
        checkpoint_id = f"checkpoint_{len(self.checkpoints)}"
        self.checkpoints[checkpoint_id] = checkpoint
        self.current_checkpoint = checkpoint_id
        print(f"Saved checkpoint: {checkpoint_id}")
        return checkpoint_id
    
    def restore_checkpoint(self, checkpoint_id):
        """Restore a previous state"""
        if checkpoint_id in self.checkpoints:
            print(f"Restoring checkpoint: {checkpoint_id}")
            return self.checkpoints[checkpoint_id]["state"]
        else:
            print(f"Checkpoint {checkpoint_id} not found")
            return None
    
    def list_checkpoints(self):
        """List all checkpoints"""
        print("=== CHECKPOINTS ===")
        for cp_id, cp_data in self.checkpoints.items():
            print(f"{cp_id}: {cp_data['label']} - {cp_data['timestamp']}")

# Example state
class WorkflowState(TypedDict):
    step: int
    data: str
    results: List[str]

# Create state manager
state_manager = StateManager()

# Simulate workflow
current_state = {
    "step": 1,
    "data": "Initial data",
    "results": []
}

print("=== STATE MANAGEMENT WITH CHECKPOINTING ===")
# Step 1
current_state["results"].append("Step 1 complete")
current_state["step"] = 2
state_manager.save_checkpoint(current_state, "After step 1")

# Step 2
current_state["data"] = "Updated data"
current_state["results"].append("Step 2 complete")
current_state["step"] = 3
state_manager.save_checkpoint(current_state, "After step 2")

# Step 3 - simulate error
current_state["results"].append("Step 3 failed!")
print(f"\nCurrent state: {current_state}")

# List checkpoints
state_manager.list_checkpoints()

# Restore from checkpoint
print("\nRestoring to checkpoint_0...")
restored_state = state_manager.restore_checkpoint("checkpoint_0")
print(f"Restored state: {restored_state}")